# **Quest 6: The Tree of Titans**

![Header image](./imgs/image_quest_6.svg)
[<img src="./imgs/instagram.svg" alt="My SVG" width="15" height="15"><small>_Monika Lipińska_</small>](https://www.instagram.com/monli_art/)

## **Part I**

On this beautiful tak‑sunny day, the tournament participants are guided to the royal gardens — specifically to the orchards where the magical potion‑trees are carefully cultivated.

Among these trees grow the **apples of superhuman strength**, fruits so potent that anyone who eats one becomes a titan. Their power depends on how they share resources through the tree’s branching structure.

### **How the trees work**

- Each fruit is represented by **`@`**
- The root is marked **`RR`**
- All other letters represent branches
- Your notes list each branch followed by the branches or fruits it connects to

Two fruits **share resources** if their paths from the root have the **same length**.  
To preserve the harvest, each tree must maintain **exactly one fruit whose path length is unique** — this fruit becomes the **most powerful**.

Your task is to determine **which fruit has the unique path length** and return the **full path from the root to that fruit**.

---

## **Example**

### **Notes**

```
RR:A,B,C
A:D,E
B:F,@
C:G,H
D:@
E:@
F:@
G:@
H:@
```

### **Tree Structure**

```
RR
├─ A ─ D ─ @
│     └ E ─ @
├─ B ─ F ─ @
│     └ @
└─ C ─ G ─ @
      └ H ─ @
```

All fruits except one lie at the same depth.  
The **only fruit with a unique path length** is reached via:

### **`RRB@`**

---

## **Your Task**

**What is the path from the root to the most powerful fruit on your tree?**

Your notes for this part appear below the puzzle on the website.


In [21]:
from collections import deque

import matplotlib.pyplot as plt
import networkx as nx
from more_itertools import first, last, one
from test_utilities import test
from util import Str

tests = [
    {
        "name": "Example",
        "s": """
            RR:A,B,C
            A:D,E
            B:F,@
            C:G,H
            D:@
            E:@
            F:@
            G:@
            H:@
        """,
        "expected": "RRB@",
    },
]


class Tree(Str):
    def __init__(self, s: str) -> None:
        self.tree = {
            first(l.strip().split(":")): last(l.strip().split(":")).split(",")
            for l in s.strip().splitlines()
        }

    def most_power_full_fruits(self, root: str = "RR") -> str:
        queue = deque([[root]])

        while queue:
            fruit_level = []
            for _ in range(len(queue)):
                path = queue.popleft()

                for child in self.tree.get(path[-1], []):
                    path1 = path + [child]
                    if child == "@":
                        fruit_level.append(self.unroll_path(path1))
                    else:
                        queue.append(path1)

            if len(fruit_level) == 1:
                return one(fruit_level)

    def unroll_path(self, path1):
        return "".join(path1)

    def _hierarchy_pos(self, G, root, width=1.0, vert_gap=0.2, vert_loc=0, xcenter=0.5):
        """
        Recursively compute positions for a tree layout.
        """
        children = list(G.successors(root))
        if not children:
            return {root: (xcenter, vert_loc)}

        dx = width / len(children)
        nextx = xcenter - width / 2 - dx / 2
        pos = {root: (xcenter, vert_loc)}

        for child in children:
            nextx += dx
            pos.update(
                self._hierarchy_pos(
                    G,
                    child,
                    width=dx,
                    vert_gap=vert_gap,
                    vert_loc=vert_loc - vert_gap,  # type: ignore
                    xcenter=nextx,
                )
            )
        return pos

    def plot(self, root="RR") -> None:
        count = 0
        edges = [
            (
                (node, neighbor)
                if neighbor != "@"
                else (node, f"{neighbor}{(count:=count+1)}")
            )
            for node, neighbors in self.tree.items()
            for neighbor in neighbors
        ]

        G = nx.DiGraph()
        G.add_edges_from(edges)

        pos = self._hierarchy_pos(G, root)

        plt.figure(figsize=(10, 8))

        options = {
            "node_size": 3000,
            "node_color": "white",
            "edgecolors": "black",
            "linewidths": 5,
            "width": 5,
            "with_labels": False,
            "arrows": False,
        }
        labels = {
            node: (f"{node[0]}" if node[0] == "@" else f"{node}") for node in G.nodes()
        }

        nx.draw(G, pos, **options)
        nx.draw_networkx_labels(G, pos, labels, font_size=36, font_color="black")

        plt.show()


@test(tests=tests[:])
def part_I(s: str) -> str:
    t = Tree(s)
    return t.most_power_full_fruits()


Test Example passed, for part_I.
Success


In [22]:
with open("../inputs/everybody_codes_e2024_q06_p1.txt") as f:
    notes1 = f.read()


print(f"Part I: {part_I(notes1)}")

Part I: RRXPPLPMGFBZ@


## **Part II**

Each participant quickly discovers the most powerful fruit on their first tree and then ventures deeper into the royal gardens, where the trees grow taller, older, and far more intricate. Once again, every knight stands before a tree bearing fruits of superhuman strength, and once again the task is the same: **identify the single fruit whose path from the root has a unique length**.

Confident in your understanding of the rules, you carefully record the complex branching structure of your new tree. The root is still marked **`RR`**, but this time the tree is far larger, with many more branches to explore.

Because the tree is so vast, the king introduces a new requirement:

### **You must describe the final path using only the _first letters_ of each branch name.**

For example, consider the path:

```
RR → ABAB → CDCD → EFEF → ROLO → @
```

The first letters of each branch name form:

```
R  A  C  E  R  @
```

So the final answer becomes:

### **`RACER@`**

---

## **Your Task**

Given your own tree notes for this part, determine:

### **What is the path — written using only the first letters of each branch — that leads from `RR` to the uniquely‑distanced, most powerful fruit?**

---


In [23]:
from test_utilities import test
from util import Str

tests = [
    {
        "name": "Example",
        "s": """
            RR:A,B,C
            A:D,E
            B:F,@
            C:G,H
            D:@
            E:@
            F:@
            G:@
            H:@
        """,
        "expected": "RB@",
    },
]


class TreeII(Tree):
    def unroll_path(self, path1):
        return "".join(map(first, path1))


@test(tests=tests[:])
def part_II(s: str) -> str:
    t = TreeII(s)
    return t.most_power_full_fruits()


Test Example passed, for part_II.
Success


In [24]:
with open("../inputs/everybody_codes_e2024_q06_p2.txt") as f:
    notes2 = f.read()

print(f"PartII: {part_II(notes2)}")

PartII: RRGXMFHJRT@


## **Part III**

The final challenge takes place at the most ancient tree in the orchard — a colossal, time‑worn giant whose branches twist across the sky and whose fruits glow with extraordinary power. This tree is older than the kingdom itself, and its structure is far more complex than anything you have encountered so far.

Unfortunately, the tree has come under attack from relentless pests: **bugs and ants**, which cleverly disguise themselves as ordinary branches. You record them in your notes along with the real connections so that the royal gardeners can deal with them later.

As before:

- The **root** is marked **`RR`**
- **Fruits** are represented by **`@`**
- All other names represent branches — including the pests
- Your task is to find the **single fruit whose path length from the root is unique**

And just like in Part II, the king requires that you describe the final path using **only the first letter of each branch name**.

For example:

```
RR → ABAB → CDCD → EFEF → ROLO → @
```

becomes:

```
RACER@
```

---

## **Your Task**

**What is the path — written using only the first letters of each branch — that leads from `RR` to the uniquely‑distanced, most powerful fruit on the ancient tree?**

---


In [25]:
with open("../inputs/everybody_codes_e2024_q06_p3.txt") as f:
    notes3 = f.read()

print(f"Part III {part_II(notes3)}")

Part III RQKZBPDZLJTS@


![happy](./imgs/happy_quack.svg)
